In [ ]:
%load_ext autoreload
%autoreload 2

from collections import Counter
import copy
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from torch.utils.data import Dataset, DataLoader

%matplotlib inline

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)


In [ ]:
df = pd.read_parquet("/Users/tomlinn/Downloads/PREPROCESSED_FLUX_TCN_INPUT.parquet")
df = df.reset_index(drop=True)



,KIC,x,label
0,757450,"[0.82121706, 0.8116605, 0.80877656, 0.7979615,...",0
1,892772,"[0.7132433, 0.6999028, 0.6930329, 0.7038331, 0...",2
2,1026032,"[1.1222867, 1.1136786, 1.1176132, 1.1167815, 1...",1
3,1026957,"[1.258851, 1.248239, 1.246251, 1.2450396, 1.24...",0
4,1027438,"[1.5406342, 1.5511838, 1.5508819, 1.5483795, 1...",2
...,...,...,...
15599,202140012,"[-2.2198217, -2.1861541, -2.1511917, -2.117524...",3
15600,202140013,"[-1.8936313, -1.8610107, -1.8271353, -1.794514...",3
15601,202140059,"[1.5995908, 1.5003388, 1.3972694, 1.2980174, 1...",2
15602,202140094,"[1.6591667, 1.6754129, 1.6922839, 1.7085301, 1...",2


In [ ]:
# Inspect the data
print(df.head())
print(df.shape)

print("\nLabel counts:")
print(df["label"].value_counts().sort_index())

lengths = df["x"].apply(len)
print("\nSequence length summary:")
print(lengths.describe())

print("\nUnique sequence lengths:", lengths.nunique())

print("\nExample x type:", type(df["x"].iloc[0]))
print("Example x first 10 values:", np.asarray(df["x"].iloc[0])[:10])
print("Example label:", df["label"].iloc[0])


       KIC                                                  x  label
0   757450  [0.82121706, 0.8116605, 0.80877656, 0.7979615,...      0
1   892772  [0.7132433, 0.6999028, 0.6930329, 0.7038331, 0...      2
2  1026032  [1.1222867, 1.1136786, 1.1176132, 1.1167815, 1...      1
3  1026957  [1.258851, 1.248239, 1.246251, 1.2450396, 1.24...      0
4  1027438  [1.5406342, 1.5511838, 1.5508819, 1.5483795, 1...      2
(15604, 3)

Label counts:
label
0    1972
1    3132
2    7520
3    2980
Name: count, dtype: int64

Sequence length summary:
count    15604.0
mean     10000.0
std          0.0
min      10000.0
25%      10000.0
50%      10000.0
75%      10000.0
max      10000.0
Name: x, dtype: float64

Unique sequence lengths: 1

Example x type: <class 'numpy.ndarray'>
Example x first 10 values: [0.82121706 0.8116605  0.80877656 0.7979615  0.78527397 0.78451854
 0.78712773 0.7833038  0.7804115  0.7763077 ]
Example label: 0


In [ ]:
assert "x" in df.columns, "Missing x column"
assert "label" in df.columns, "Missing label column"

assert df["label"].notna().all(), "Found missing labels"
assert df["x"].notna().all(), "Found missing sequences"

sample0 = df["x"].iloc[0]
assert hasattr(sample0, "__len__"), "x does not look like a sequence"

lengths = df["x"].apply(len)
assert lengths.nunique() == 1, (
    f"Sequences have multiple lengths: {lengths.value_counts().head()}. "
    "This notebook expects equal-length sequences."
)

print("Input checks passed.")


Input checks passed.


In [ ]:
# Split Function
def splitData(df, test_size=0.2, val_size=0.2, random_state=42):
    train_df, temp_df = train_test_split(
        df,
        test_size=test_size,
        random_state=random_state,
        stratify=df["label"]
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        sampler=sampler,
        drop_last=False
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False
    )

    return train_loader, val_loader, test_loader


In [ ]:
# Dataset Class
class ArrayDataset(Dataset):
    def __init__(self, df):
        self.X = df["x"].tolist()
        self.y = df["label"].to_numpy(dtype=np.int64)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = np.asarray(self.X[idx], dtype=np.float32)

        if not np.all(np.isfinite(x)):
            raise ValueError(f"Non-finite values found in sample {idx}")

        x = torch.tensor(x, dtype=torch.float32).unsqueeze(0)  # (1, T)
        y = torch.tensor(self.y[idx], dtype=torch.long)
        return x, y


In [ ]:
def compute_metrics(y_true, y_pred, labels=None, target_names=None):
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "weighted_recall": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }

    report = classification_report(
        y_true,
        y_pred,
        labels=labels,
        target_names=target_names,
        zero_division=0
    )

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    return metrics, report, cm


def evaluate_model(model, loader, device, labels=None, target_names=None):
    model.eval()
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            logits = model(x)
            preds = logits.argmax(dim=1).cpu().numpy()

            all_preds.extend(preds)
            all_targets.extend(y.numpy())

    metrics, report, cm = compute_metrics(
        all_targets,
        all_preds,
        labels=labels,
        target_names=target_names
    )

    return metrics, report, cm, all_targets, all_preds


In [ ]:
# Metrics and Evaluation
def compute_metrics(y_true, y_pred, labels=None, target_names=None):
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "weighted_recall": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }

    report = classification_report(
        y_true,
        y_pred,
        labels=labels,
        target_names=target_names,
        zero_division=0
    )

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    return metrics, report, cm


def evaluate_model(model, loader, device, labels=None, target_names=None):
    model.eval()
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            logits = model(x)
            preds = logits.argmax(dim=1).cpu().numpy()

            all_preds.extend(preds)
            all_targets.extend(y.numpy())

    metrics, report, cm = compute_metrics(
        all_targets,
        all_preds,
        labels=labels,
        target_names=target_names
    )

    return metrics, report, cm, all_targets, all_preds


In [ ]:
# TCN Model
class TCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=5, dilation=1, dropout=0.1):
        super().__init__()
        padding = (kernel_size - 1) * dilation // 2

        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size, padding=padding, dilation=dilation)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size, padding=padding, dilation=dilation)
        self.dropout = nn.Dropout(dropout)
        self.residual = nn.Conv1d(in_ch, out_ch, kernel_size=1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        res = self.residual(x)
        x = F.relu(self.conv1(x))
        x = self.dropout(x)
        x = F.relu(self.conv2(x))
        x = self.dropout(x)
        return F.relu(x + res)


In [ ]:
# TCN Classifier
class TCNClassifier(nn.Module):
    def __init__(self, num_classes, in_ch=1, hidden=64, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            TCNBlock(in_ch, hidden, dilation=1, dropout=dropout),
            TCNBlock(hidden, hidden * 2, dilation=2, dropout=dropout),
            TCNBlock(hidden * 2, hidden * 4, dilation=4, dropout=dropout),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(hidden * 4, num_classes)

    def forward(self, x):
        x = self.net(x)
        x = self.pool(x).squeeze(-1)
        return self.fc(x)


In [ ]:
# Split the data and create DataLoaders
train_df, val_df, test_df = splitData(df)
print("Split sizes:", len(train_df), len(val_df), len(test_df))

train_loader, val_loader, test_loader = makeDataLoaders(train_df, val_df, test_df, batch_size=32)


Split sizes: 12483 2340 781


In [ ]:
print("\n--- DataLoader sanity check ---")

ds = ArrayDataset(train_df)
x0, y0 = ds[0]

print("Single sample x shape:", x0.shape)
print("Single sample y:", y0)

assert x0.ndim == 2
assert x0.shape[0] == 1
assert torch.isfinite(x0).all()

x_batch, y_batch = next(iter(train_loader))

print("Batch x shape:", x_batch.shape)
print("Batch y shape:", y_batch.shape)

assert x_batch.ndim == 3
assert x_batch.shape[1] == 1
assert y_batch.ndim == 1
assert torch.isfinite(x_batch).all()

print("Batch mean:", x_batch.mean().item())
print("Batch std:", x_batch.std().item())

assert x_batch.std().item() > 0
print("DataLoader sanity check passed.\n")



--- DataLoader sanity check ---
Single sample x shape: torch.Size([1, 10000])
Single sample y: tensor(1)
Batch x shape: torch.Size([32, 1, 10000])
Batch y shape: torch.Size([32])
Batch mean: 1.5258788677030566e-09
Batch std: 1.0000015497207642
DataLoader sanity check passed.



In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

class_names = [
    "CONFIRMED",
    "ECLIPSING BINARY STAR",
    "FALSE POSITIVE",
    "VARIABLE STAR"
]
labels = [0, 1, 2, 3]

num_classes = df["label"].nunique()
model = TCNClassifier(num_classes=num_classes, in_ch=1, hidden=64, dropout=0.1).to(device)

print("Device:", device)
print("Num classes:", num_classes)


Device: mps
Num classes: 4


In [ ]:
counts = train_df["label"].value_counts().sort_index()
weights_np = (counts.sum() / (len(counts) * counts)).to_numpy()
weights = torch.tensor(weights_np, dtype=torch.float32, device=device)

criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

print("Class counts:", counts.to_dict())
print("Alpha weights:", alpha.detach().cpu().numpy())


Class counts: {0: 1578, 1: 2505, 2: 6016, 3: 2384}
Alpha weights: [1.0962536  0.9865345  0.41078272 1.0366061 ]


In [ ]:
print("\n--- Model / Device sanity check ---")

param_device = next(model.parameters()).device
print("Model parameters device:", param_device)
assert param_device.type == device.type

x_batch, y_batch = next(iter(train_loader))
x_batch = x_batch.to(device)
y_batch = y_batch.to(device)

with torch.no_grad():
    logits = model(x_batch)
    preds = logits.argmax(dim=1)

print("Logits shape:", logits.shape)
print("Logits device:", logits.device)
print("Preds unique:", torch.unique(preds, return_counts=True))

assert logits.shape[0] == x_batch.shape[0]
assert logits.shape[1] == num_classes
assert torch.isfinite(logits).all()

print("Logits sample:", logits[0].detach().cpu().numpy())
print("Model/device sanity check passed.\n")



--- Model / Device sanity check ---
Model parameters device: mps:0
Logits shape: torch.Size([32, 4])
Logits device: mps:0
Preds unique: (tensor([2], device='mps:0'), tensor([32], device='mps:0'))
Logits sample: [ 0.02884265 -0.0057724   0.09574314  0.00132615]
Model/device sanity check passed.



In [ ]:
# Gradient Check
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

x_batch, y_batch = next(iter(train_loader))
x_batch = x_batch.to(device)
y_batch = y_batch.to(device)

optimizer.zero_grad()
logits = model(x_batch)
loss = criterion(logits, y_batch)
loss.backward()

printed = False
for name, param in model.named_parameters():
    if param.grad is not None:
        print(name, "grad norm:", param.grad.norm().item())
        printed = True
        break

if not printed:
    print("No gradients found.")


net.0.conv1.weight grad norm: 0.0025414240080863237


In [ ]:
print("\n--- Overfitting sanity check ---")

model = TCNClassifier(num_classes=num_classes, in_ch=1, hidden=64, dropout=0.1).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
criterion_small = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

x_small, y_small = next(iter(train_loader))
x_small = x_small[:8].to(device)
y_small = y_small[:8].to(device)

losses = []

for step in range(100):
    optimizer_overfit.zero_grad()
    logits = model_overfit(x_small)
    loss = criterion_overfit(logits, y_small)
    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    losses.append(loss.item())

    if step % 10 == 0:
        preds = logits.argmax(dim=1)
        acc = (preds == y_small).float().mean().item()
        print(f"Step {step:02d} | loss {loss.item():.4f} | acc {acc:.4f}")

print("Initial loss:", losses[0])
print("Final loss:", losses[-1])

assert losses[-1] < losses[0], "Tiny-batch overfit test failed"
print("Overfit sanity check passed.\n")



--- Overfitting sanity check ---
Step 00 | loss 1.4340 | acc 0.0000
Step 10 | loss 1.1353 | acc 0.7500
Step 20 | loss 1.0424 | acc 0.7500
Step 30 | loss 0.7050 | acc 1.0000
Step 40 | loss 0.5011 | acc 0.8750
Step 50 | loss 0.5018 | acc 0.8750
Step 60 | loss 0.4358 | acc 1.0000
Step 70 | loss 0.4218 | acc 1.0000
Step 80 | loss 0.4117 | acc 1.0000
Step 90 | loss 0.4019 | acc 1.0000
Initial loss: 1.4340059757232666
Final loss: 0.3917832374572754
Overfit sanity check passed.



In [ ]:
model = TCNClassifier(num_classes=num_classes, in_ch=1, hidden=64, dropout=0.1).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

best_val_macro_f1 = -1.0
best_state = copy.deepcopy(model.state_dict())
num_epochs = 20

for epoch in range(1, num_epochs + 1):
    model.train()
    train_loss = 0.0
    train_preds = []
    train_targets = []

    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item() * x.size(0)
        train_preds.extend(logits.argmax(dim=1).detach().cpu().numpy())
        train_targets.extend(y.detach().cpu().numpy())

    train_loss /= len(train_loader.dataset)
    train_metrics, _, _ = compute_metrics(
        train_targets,
        train_preds,
        labels=labels,
        target_names=class_names
    )

    val_metrics, val_report, val_cm, _, _ = evaluate_model(
        model, val_loader, device, labels=labels, target_names=class_names
    )

    scheduler.step(val_metrics["macro_f1"])

    print(
        f"Epoch {epoch:02d} | "
        f"TRAIN acc {train_metrics['accuracy']:.4f} "
        f"bal acc {train_metrics['balanced_accuracy']:.4f} "
        f"macro f1 {train_metrics['macro_f1']:.4f} | "
        f"VAL acc {val_metrics['accuracy']:.4f} "
        f"bal acc {val_metrics['balanced_accuracy']:.4f} "
        f"macro f1 {val_metrics['macro_f1']:.4f}"
    )

    if val_metrics["macro_f1"] > best_val_macro_f1:
        best_val_macro_f1 = val_metrics["macro_f1"]
        best_state = copy.deepcopy(model.state_dict())
        torch.save(best_state, "best_tcn.pt")
        print("  saved best_tcn.pt")


Epoch 01 | train loss 1.0673 acc 0.4279 f1 0.4593 | val loss 1.0015 acc 0.4299 f1 0.4478
  saved best_tcn.pt
Epoch 02 | train loss 0.9329 acc 0.5008 f1 0.5346 | val loss 0.9008 acc 0.4684 f1 0.4930
  saved best_tcn.pt
Epoch 03 | train loss 0.8661 acc 0.5684 f1 0.5945 | val loss 0.8547 acc 0.6175 f1 0.6346
  saved best_tcn.pt
Epoch 04 | train loss 0.8329 acc 0.5939 f1 0.6186 | val loss 0.8366 acc 0.6269 f1 0.6463
  saved best_tcn.pt
Epoch 05 | train loss 0.8039 acc 0.6068 f1 0.6304 | val loss 0.8088 acc 0.6308 f1 0.6496
  saved best_tcn.pt
Epoch 06 | train loss 0.7951 acc 0.6095 f1 0.6331 | val loss 0.8340 acc 0.6115 f1 0.6325
Epoch 07 | train loss 0.7767 acc 0.6195 f1 0.6421 | val loss 0.8058 acc 0.6355 f1 0.6530
  saved best_tcn.pt
Epoch 08 | train loss 0.7773 acc 0.6171 f1 0.6401 | val loss 0.7914 acc 0.6295 f1 0.6502
  saved best_tcn.pt
Epoch 09 | train loss 0.7578 acc 0.6279 f1 0.6508 | val loss 0.7856 acc 0.6423 f1 0.6602
  saved best_tcn.pt
Epoch 10 | train loss 0.7569 acc 0.6304

In [ ]:
model.load_state_dict(best_state)
model.eval()
print("Loaded best model.")


Loaded best model.


In [ ]:
test_metrics, test_report, test_cm, y_true, y_pred = evaluate_model(
    model, test_loader, device, labels=labels, target_names=class_names
)

print("Overall metrics:")
print(f"Accuracy:            {test_metrics['accuracy']:.4f}")
print(f"Balanced accuracy:   {test_metrics['balanced_accuracy']:.4f}")
print(f"Macro precision:     {test_metrics['macro_precision']:.4f}")
print(f"Macro recall:        {test_metrics['macro_recall']:.4f}")
print(f"Macro F1:            {test_metrics['macro_f1']:.4f}")
print(f"Weighted precision:   {test_metrics['weighted_precision']:.4f}")
print(f"Weighted recall:      {test_metrics['weighted_recall']:.4f}")
print(f"Weighted F1:          {test_metrics['weighted_f1']:.4f}")

print("\nPer-class report:")
print(test_report)

print("Confusion matrix:")
print(test_cm)


Overall metrics:
Accuracy:            0.6671
Balanced accuracy:   0.7718
Macro precision:     0.6894
Macro recall:        0.7718
Macro F1:            0.6850
Weighted precision:   0.7545
Weighted recall:      0.6671
Weighted F1:          0.6652

Per-class report:
                       precision    recall  f1-score   support

            CONFIRMED       0.41      0.95      0.57        99
ECLIPSING BINARY STAR       0.89      0.89      0.89       157
       FALSE POSITIVE       0.84      0.45      0.58       376
        VARIABLE STAR       0.62      0.81      0.70       149

             accuracy                           0.67       781
            macro avg       0.69      0.77      0.68       781
         weighted avg       0.75      0.67      0.67       781

Confusion matrix:
[[ 94   4   1   0]
 [ 13 139   2   3]
 [124  12 168  72]
 [  0   1  28 120]]


In [ ]:
train_metrics, train_report, train_cm, train_true, train_pred = evaluate_model(
    model, train_loader, device, labels=labels, target_names=class_names
)

val_metrics, val_report, val_cm, val_true, val_pred = evaluate_model(
    model, val_loader, device, labels=labels, target_names=class_names
)

test_metrics, test_report, test_cm, test_true, test_pred = evaluate_model(
    model, test_loader, device, labels=labels, target_names=class_names
)

print("===== FINAL RESULTS =====")

print("\nTRAIN metrics:")
print(f"Accuracy:            {train_metrics['accuracy']:.4f}")
print(f"Balanced accuracy:   {train_metrics['balanced_accuracy']:.4f}")
print(f"Macro precision:     {train_metrics['macro_precision']:.4f}")
print(f"Macro recall:        {train_metrics['macro_recall']:.4f}")
print(f"Macro F1:            {train_metrics['macro_f1']:.4f}")
print(f"Weighted precision:  {train_metrics['weighted_precision']:.4f}")
print(f"Weighted recall:     {train_metrics['weighted_recall']:.4f}")
print(f"Weighted F1:         {train_metrics['weighted_f1']:.4f}")

print("\nVAL metrics:")
print(f"Accuracy:            {val_metrics['accuracy']:.4f}")
print(f"Balanced accuracy:   {val_metrics['balanced_accuracy']:.4f}")
print(f"Macro precision:     {val_metrics['macro_precision']:.4f}")
print(f"Macro recall:        {val_metrics['macro_recall']:.4f}")
print(f"Macro F1:            {val_metrics['macro_f1']:.4f}")
print(f"Weighted precision:  {val_metrics['weighted_precision']:.4f}")
print(f"Weighted recall:     {val_metrics['weighted_recall']:.4f}")
print(f"Weighted F1:         {val_metrics['weighted_f1']:.4f}")

print("\nTEST metrics:")
print(f"Accuracy:            {test_metrics['accuracy']:.4f}")
print(f"Balanced accuracy:   {test_metrics['balanced_accuracy']:.4f}")
print(f"Macro precision:     {test_metrics['macro_precision']:.4f}")
print(f"Macro recall:        {test_metrics['macro_recall']:.4f}")
print(f"Macro F1:            {test_metrics['macro_f1']:.4f}")
print(f"Weighted precision:  {test_metrics['weighted_precision']:.4f}")
print(f"Weighted recall:     {test_metrics['weighted_recall']:.4f}")
print(f"Weighted F1:         {test_metrics['weighted_f1']:.4f}")

print("\nPer-class report (TEST):")
print(test_report)

print("Confusion matrix (TEST):")
print(test_cm)



Test accuracy: 0.6671
Test macro F1: 0.685

Classification report:

              precision    recall  f1-score   support

           0       0.41      0.95      0.57        99
           1       0.89      0.89      0.89       157
           2       0.84      0.45      0.58       376
           3       0.62      0.81      0.70       149

    accuracy                           0.67       781
   macro avg       0.69      0.77      0.68       781
weighted avg       0.75      0.67      0.67       781

Confusion matrix:

[[ 94   4   1   0]
 [ 13 139   2   3]
 [124  12 168  72]
 [  0   1  28 120]]


In [ ]:
x_batch, y_batch = next(iter(train_loader))
x_batch = x_batch.to(device)

with torch.no_grad():
    preds = model(x_batch).argmax(dim=1)

print("Pred classes:", torch.unique(preds, return_counts=True))
print("True classes:", torch.unique(y_batch, return_counts=True))
print("First 20 preds:", preds[:20].cpu().numpy())
print("First 20 labels:", y_batch[:20].cpu().numpy())


Pred classes: (tensor([0, 1, 2, 3], device='mps:0'), tensor([9, 7, 7, 9], device='mps:0'))
True classes: (tensor([0, 1, 2, 3]), tensor([ 3,  7, 17,  5]))
First 20 preds: [2 1 1 3 3 1 1 2 0 3 3 3 0 3 3 0 2 2 2 0]
First 20 labels: [3 1 1 3 2 1 1 2 2 3 2 2 2 3 2 2 2 2 2 0]


In [ ]:
cm_df = pd.DataFrame(
    test_cm,
    index=class_names,
    columns=class_names
)
cm_df


,CONFIRMED,ECLIPSING BINARY STAR,FALSE POSITIVE,VARIABLE STAR
CONFIRMED,93,4,0,2
ECLIPSING BINARY STAR,14,131,3,9
FALSE POSITIVE,129,11,127,109
VARIABLE STAR,0,0,12,137


In [ ]:
print("\nGeneralization gap (train vs test accuracy):",
      round(train_metrics["accuracy"] - test_metrics["accuracy"], 4))



Generalization gap (train vs test accuracy): 0.0508
